# 06 — Prophet & Probabilistic Forecasting

LightGBM is a powerful function approximator but it produces only **point
forecasts**. Many business decisions need a forecast *distribution* — what's
the worst case at the 90th percentile? — so we now bring in two complementary
probabilistic approaches.

### What is Prophet?
Prophet decomposes a time series into:
$$ y(t) = g(t) + s(t) + h(t) + \varepsilon_t $$

- **g(t)** — piecewise-linear or logistic trend with automatic changepoint
  detection
- **s(t)** — Fourier-based seasonality (yearly, weekly, daily, custom)
- **h(t)** — holiday effects (built-in country calendars + your own)
- **ε** — Gaussian noise; uncertainty intervals are produced via Monte-Carlo
  simulation of the posterior

### Why use it?
1. **Fully probabilistic** — every forecast comes with `yhat_lower` /
   `yhat_upper`.
2. **Interpretable by construction** — you can plot trend / weekly /
   yearly components separately.
3. **Robust to messy data** — handles missing values and holidays gracefully.
4. **Sensible defaults** — reasonable forecasts with zero tuning.

### Limitations
- Independent model per series → cost grows linearly with the number of keys.
- No automatic cross-series learning (vs. LightGBM's shared structure).
- Lag-based features must be encoded as exogenous regressors manually.

We'll fit Prophet through `MultiKeyProphet`, our wrapper that loops over keys,
trains one model each, and concatenates the per-key predictions.


## A pause: how does Prophet actually work?

If LightGBM is a committee of analysts each fixing the previous one's
mistakes, **Prophet is a single careful analyst** who decomposes the series
into named, interpretable components and fits them all at once.

### The model

$$ y(t) = g(t) + s(t) + h(t) + \varepsilon_t $$

| Component | What it captures | The business reading |
|---|---|---|
| $g(t)$ — **trend** | Long-term direction, with automatic detection of "changepoints" where growth shifts gear | *"Sales started accelerating in Q3 of last year"* |
| $s(t)$ — **seasonality** | Repeating patterns: weekly cycle, yearly cycle, optionally daily or custom | *"Saturdays are 35% above the weekly average; July peaks 20% above the yearly average"* |
| $h(t)$ — **holidays** | Bumps or dips around named events (Christmas, Diwali, Black Friday) | *"Diwali week adds ~₹120K incremental sales"* |
| $\varepsilon_t$ — **noise** | What's left over after the structured components | *"Day-to-day randomness — the irreducible variability"* |

### How each piece is parameterised

- **Trend** is *piecewise linear* (or logistic for capped growth): a series
  of line segments connected at automatically-detected changepoints. Prophet
  proposes 25 candidate changepoints across history and lets the data decide
  which ones matter (via a sparse prior — most get shrunk to zero).
- **Seasonality** uses a *Fourier series* — sums of sines and cosines. A
  small number of Fourier terms captures smooth seasonal shapes; more terms
  capture sharper ones. You control the flexibility via
  `yearly_seasonality=10`, `weekly_seasonality=3`, etc.
- **Holidays** are simple impulse effects: each holiday adds a learned
  constant on its date (and optionally on a window of days before/after).
- **Uncertainty intervals** are generated by sampling *future trend
  changepoints* from the same distribution that fit the historical data —
  which is why the bands get wider as you forecast further out (the model
  is honest about not knowing when the next regime change will occur).

### Prophet's strengths

1. **Every forecast is a story.** *"The forecast for next month is up 8%,
   driven by the seasonal lift (+12%) and the underlying trend (+3%),
   partially offset by the absence of last month's holiday effect (-7%)."*
   Try delivering that explanation from a black-box model.
2. **Probabilistic by default.** Every prediction is a band, not a point.
   That maps directly to inventory safety stock and staffing buffer
   decisions.
3. **Sensible without tuning.** A first cut works on most well-behaved
   business series. The defaults reflect lessons from running this model
   on tens of thousands of internal time series at Facebook.
4. **Robust to messy reality.** Missing dates, outliers, abrupt level
   shifts, and large gaps are all tolerated more gracefully than by
   ARIMA-family models.

### limitations

- **One model per series.** No cross-series learning — if you have 50,000
  SKUs, training and inference cost grows linearly. LightGBM trains one
  model across all series and learns shared patterns.
- **Limited interactions.** The additive structure can't easily express
  *"the weekly seasonality is bigger during the holiday season"* without
  manual feature engineering. LightGBM picks this up automatically.
- **Lag features need to be added as regressors.** Prophet doesn't
  natively use yesterday's sales the way LightGBM does — you bolt them on
  via `add_regressor`.
- **Trend extrapolation can be over-confident** on short or recently
  volatile series. Prophet trusts the most recent slope unless you cap it
  with a logistic ceiling.

### When to reach for which model

| Situation | Lean toward |
|---|---|
| Many series with shared structure (substitution, cannibalisation, store-format effects) | **LightGBM** |
| You need a clean per-component story for a stakeholder presentation | **Prophet** |
| Need calibrated uncertainty intervals out of the box | **Prophet** (or LightGBM-Quantile in notebook 08) |
| Rich exogenous features (price, promotions, weather) drive demand | **LightGBM** |
| Stable historical seasonality with named holiday effects | **Prophet** |
| Daily retraining on millions of rows | **LightGBM** (Prophet is slower per series) |
| Brand-new series with only a few months of data | **Prophet** (graceful with limited history) |

In practice, **run both** - LightGBM for accuracy with SHAP for
diagnostics, Prophet alongside for the narrative. They're complementary,
not competitors.


In [1]:
# === Colab / local setup ====================================================
# 1. Install dependencies (uncomment the pip line on first Colab run).
# !pip install -q lightgbm==4.* prophet plotly optuna shap pandas numpy scikit-learn pyarrow

# 2. Make the `utils` package importable. Two options:
#    (a) Notebook is sitting next to a `utils/` folder (recommended).
#    (b) The package is uploaded as a zip; unzip it and `sys.path.append(...)`.
import os, sys
HERE = os.path.dirname(os.path.abspath("__file__"))  # may be empty in Colab
for cand in [".", "..", "/content", "/content/ml_forecasting_tutorial"]:
    if os.path.isdir(os.path.join(cand, "utils")):
        sys.path.insert(0, cand)
        break

# 3. Standard imports for every notebook.
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "colab"   # works in Colab + Jupyter

# 4. Tell pandas to display nicely.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


In [2]:
# === Dataset configuration ==================================================
# EDIT THIS CELL TO POINT AT YOUR DATASET
#DATA_PATH    = "/content/sales.csv"             # path to your Kaggle CSV / parquet
#DATE_COL     = "date"                           # date column
#TARGET_COL   = "sales"                          # target / forecast column
#KEY_COLS     = ["store", "item"]                # columns identifying a unique series
#EXCLUDE_COLS = []
#FREQ         = "D"                              # 'D'=daily, 'W'=weekly, 'M'=monthly
#HOLDOUT_DAYS = 28                               # length of test horizon

DATA_PATH    = "./dataset/m5/m5_tiny.csv"             # path to your CSV / parquet
DATE_COL     = "date"                              # date column
TARGET_COL   = "sales"                          # target / forecast column
KEY_COLS     = ['item_id',
                'dept_id',
                'cat_id',
                'store_id',
                'state_id']                           # columns identifying a unique series
EXCLUDE_COLS = []
FREQ         = "D"                              # 'D'=daily, 'W'=weekly, 'M'=monthly
HOLDOUT_DAYS = 28                               # length of test horizon

## 1. Load data and select a manageable sample

Prophet trains one model per key, so we'll pick a handful of high-volume keys for an interactive demo. In production you'd batch-fit all keys (possibly in parallel).

In [3]:
from utils.data_utils import load_forecasting_data, complete_panel, time_based_split

raw = load_forecasting_data(DATA_PATH, DATE_COL, TARGET_COL, KEY_COLS)
panel = complete_panel(raw, DATE_COL, KEY_COLS, TARGET_COL, freq=FREQ, fill_value=0.0)
cutoff = panel[DATE_COL].max() - pd.Timedelta(days=HOLDOUT_DAYS - 1)
train_df, test_df = time_based_split(panel, DATE_COL, cutoff=cutoff)

# Pick the 6 keys with the highest training volume — easier to visualise.
top_keys = (train_df.groupby(KEY_COLS)[TARGET_COL].sum()
            .sort_values(ascending=False).head(6).index)
mask_train = train_df.set_index(KEY_COLS).index.isin(top_keys)
mask_test  = test_df.set_index(KEY_COLS).index.isin(top_keys)
train_sub  = train_df[mask_train].copy()
test_sub   = test_df[mask_test].copy()
print("selected keys:", list(top_keys))
print("rows:", len(train_sub), "train,", len(test_sub), "test")


selected keys: [('FOODS_3_377', 'FOODS_3', 'FOODS', 'TX_3', 'TX'), ('FOODS_3_694', 'FOODS_3', 'FOODS', 'WI_2', 'WI'), ('FOODS_3_202', 'FOODS_3', 'FOODS', 'TX_2', 'TX'), ('FOODS_3_681', 'FOODS_3', 'FOODS', 'TX_3', 'TX'), ('FOODS_3_586', 'FOODS_3', 'FOODS', 'WI_1', 'WI'), ('FOODS_3_607', 'FOODS_3', 'FOODS', 'TX_2', 'TX')]
rows: 2652 train, 168 test


## 2. Fit Prophet — one model per key

Prophet's defaults already give you yearly + weekly seasonality and a
piecewise-linear trend. We'll add `country_holidays="US"` (change to your
locale!) to capture public-holiday effects.

`interval_width=0.80` says we want an 80% prediction band. Set
`mcmc_samples=300` (slow!) for full Bayesian posterior intervals; the default
MAP fit gives parametric intervals and is much faster.


In [4]:
from utils.prophet_forecaster import MultiKeyProphet

prophet_kwargs = dict(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode="multiplicative",   # try "additive" if your data has stable amplitude
    changepoint_prior_scale=0.05,        # higher → more flexible trend
    interval_width=0.80,
)
mkp = MultiKeyProphet(
    date_col=DATE_COL,
    target_col=TARGET_COL,
    key_cols=KEY_COLS,
    prophet_kwargs=prophet_kwargs,
    country_holidays="US",
    floor_zero=True,
)
mkp.fit(train_sub, verbose=True)


22:40:18 - cmdstanpy - INFO - Chain [1] start processing
22:40:19 - cmdstanpy - INFO - Chain [1] done processing
22:40:19 - cmdstanpy - INFO - Chain [1] start processing
22:40:19 - cmdstanpy - INFO - Chain [1] done processing
22:40:19 - cmdstanpy - INFO - Chain [1] start processing
22:40:19 - cmdstanpy - INFO - Chain [1] done processing
22:40:19 - cmdstanpy - INFO - Chain [1] start processing
22:40:19 - cmdstanpy - INFO - Chain [1] done processing
22:40:19 - cmdstanpy - INFO - Chain [1] start processing
22:40:19 - cmdstanpy - INFO - Chain [1] done processing


  fit Prophet for key=('FOODS_3_202', 'FOODS_3', 'FOODS', 'TX_2', 'TX')  (n=442)
  fit Prophet for key=('FOODS_3_377', 'FOODS_3', 'FOODS', 'TX_3', 'TX')  (n=442)
  fit Prophet for key=('FOODS_3_586', 'FOODS_3', 'FOODS', 'WI_1', 'WI')  (n=442)
  fit Prophet for key=('FOODS_3_607', 'FOODS_3', 'FOODS', 'TX_2', 'TX')  (n=442)
  fit Prophet for key=('FOODS_3_681', 'FOODS_3', 'FOODS', 'TX_3', 'TX')  (n=442)


22:40:19 - cmdstanpy - INFO - Chain [1] start processing
22:40:19 - cmdstanpy - INFO - Chain [1] done processing


  fit Prophet for key=('FOODS_3_694', 'FOODS_3', 'FOODS', 'WI_2', 'WI')  (n=442)


## 3. Forecast the holdout period

Prophet predicts on a date list — it doesn't need lagged history at inference
time. The output frame contains `yhat`, `yhat_lower`, `yhat_upper` for each
date and key.


In [5]:
future_dates = sorted(test_sub[DATE_COL].unique())
proph_pred = mkp.predict(future_dates=future_dates)
proph_pred.head()


,date,item_id,dept_id,cat_id,store_id,state_id,yhat,yhat_lower,yhat_upper
0,2016-05-23,FOODS_3_202,FOODS_3,FOODS,TX_2,TX,23.810628,-0.400820,46.844939
1,2016-05-24,FOODS_3_202,FOODS_3,FOODS,TX_2,TX,24.315953,-0.089717,46.441870
2,2016-05-25,FOODS_3_202,FOODS_3,FOODS,TX_2,TX,18.224403,-5.843624,43.014428
3,2016-05-26,FOODS_3_202,FOODS_3,FOODS,TX_2,TX,22.790891,-2.232851,47.108455
4,2016-05-27,FOODS_3_202,FOODS_3,FOODS,TX_2,TX,22.592686,-1.109660,47.821466


## 4. Visualise the probabilistic forecast

`plot_forecast` draws history + actual + forecast + prediction band for one key.
We'll loop over our selected keys.


In [6]:
from utils.viz import plot_forecast

key0 = list(top_keys)[0]
key_filter = (test_sub.set_index(KEY_COLS).index == key0)
hist_filter = (train_sub.set_index(KEY_COLS).index == key0)
pred_filter = (proph_pred.set_index(KEY_COLS).index == key0)

fig = plot_forecast(
    history=train_sub[hist_filter].tail(180),
    actual=test_sub[key_filter],
    predicted=proph_pred[pred_filter],
    date_col=DATE_COL, target_col=TARGET_COL,
    yhat_col="yhat", yhat_lower_col="yhat_lower", yhat_upper_col="yhat_upper",
    title=f"Prophet forecast — key={key0}",
)
fig.show()


## 5. Component decomposition — interpretability for free

Where Prophet shines: every fitted model can hand back its trend, weekly
seasonality, yearly seasonality and holiday components separately. Showing
these to a stakeholder turns a forecast into a *story*: "demand has been
trending up since March, weekends amplify it 30%, and Christmas is the big
yearly spike."


In [7]:
from utils.explainer import prophet_decomposition
from utils.viz import plot_prophet_components

# Pick one key
key0_tuple = key0 if isinstance(key0, tuple) else (key0,)
decomp = prophet_decomposition(mkp.models[key0_tuple], key_label=str(key0))
fig = plot_prophet_components(decomp, title=f"Prophet decomposition — key={key0}")
fig.show()


## 6. Probabilistic evaluation

For probabilistic forecasts we need *probabilistic metrics* in addition to
point metrics:

- **Pinball loss** at quantile q — the proper scoring rule for that quantile.
- **Coverage** — fraction of actuals that fall within the prediction band; for
  an 80% band, well-calibrated coverage should be ≈ 0.80. Lower means the
  intervals are too narrow (over-confident).
- **WAPE** of the median is still useful as a point-accuracy summary.


In [8]:
from utils.metrics import pinball_loss, coverage, wape, metric_report
import numpy as np

merged = test_sub.merge(proph_pred, on=[DATE_COL, *KEY_COLS], how="left")

y_true  = merged[TARGET_COL].values
y_hat   = merged["yhat"].values
y_lo    = merged["yhat_lower"].values
y_hi    = merged["yhat_upper"].values

print(f"WAPE (median)           : {wape(y_true, y_hat):.2f}")
print(f"Pinball@10              : {pinball_loss(y_true, y_lo, q=0.10):.3f}")
print(f"Pinball@90              : {pinball_loss(y_true, y_hi, q=0.90):.3f}")
print(f"Empirical 80% coverage  : {coverage(y_true, y_lo, y_hi):.2%}")


WAPE (median)           : nan
Pinball@10              : 11.960
Pinball@90              : 3.437
Empirical 80% coverage  : 27.38%


### Interpreting the numbers
- If empirical coverage is much **below** 80%, your intervals are too tight.
  Increase `interval_width`, raise `changepoint_prior_scale`, or switch to
  `mcmc_samples > 0` for a full Bayesian posterior.
- If coverage is much **above** 80%, the bands are too wide and you're paying
  a real cost in inventory / staffing. Tighten the priors.


## 7. Adding exogenous regressors

Prophet can ingest external variables via `add_regressor`. Our wrapper exposes
this through `extra_regressors=[(name, prior_scale, mode), …]`.

A common trick: encode marketing spend, price, or weather as regressors and let
Prophet learn their effects. The skeleton below is commented out — uncomment
and adapt to your dataset.


In [9]:
# Skeleton (uncomment + adapt):
# extra = [("price", 5.0, "additive"), ("promo_flag", 5.0, "multiplicative")]
# mkp_x = MultiKeyProphet(
#     date_col=DATE_COL, target_col=TARGET_COL, key_cols=KEY_COLS,
#     prophet_kwargs=prophet_kwargs, country_holidays="US",
#     extra_regressors=extra,
# )
# mkp_x.fit(train_sub_with_regressors)
# future_pred = mkp_x.predict(future_dates=future_dates,
#                             exogenous_future=test_sub_with_regressors)


## Recap

- Prophet fits an additive model per key with built-in trend, seasonality,
  and holiday effects, and emits prediction intervals natively.
- We measured both point accuracy (WAPE) and probabilistic quality (pinball,
  coverage). The latter tell us whether the *uncertainty* is well-calibrated.
- Component decomposition makes Prophet trivially interpretable.
- Use it when the series have stable seasonality and you need uncertainty —
  use LightGBM when you need cross-series learning, sharp non-linear
  interactions, or rich exogenous features.

In **notebook 07** we'll dive into explainability: gain/split importance and
SHAP values for the LightGBM models, plus a deeper look at the Prophet
components.
